# Multi-Image Product Angle Generation

Pipeline para gerar imagens de produtos em diferentes angulos a partir de 1 ou mais fotos de referencia.

**Modelos utilizados:**
- Base: [Qwen/Qwen-Image-Edit-2509](https://huggingface.co/Qwen/Qwen-Image-Edit-2509)
- LoRA: [dx8152/Qwen-Edit-2509-Multiple-angles](https://huggingface.co/dx8152/Qwen-Edit-2509-Multiple-angles)

**Capacidades:**
- Aceita 1 ou mais imagens de entrada do mesmo produto
- Gera novas imagens em angulos variados (frontal, lateral, top-down, wide-angle, close-up, etc.)
- Controle preciso de camera virtual (rotacao, elevacao, distancia)
- Ideal para marketplaces e e-commerce

## 1. Setup

In [ ]:
!pip install "diffusers>=0.32.0,<1.0.0" transformers accelerate pillow

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 2. Configuracao

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*Flax classes are deprecated.*")

import torch
from pathlib import Path
from PIL import Image
from diffusers import QwenImageEditPlusPipeline
from diffusers.utils import load_image

# --- Configuracao ---
HF_BASE_MODEL = "Qwen/Qwen-Image-Edit-2509"
LORA_ANGLES = "dx8152/Qwen-Edit-2509-Multiple-angles"

DEVICE = "cuda"  # "mps" para Apple Silicon
DTYPE = torch.bfloat16

OUTPUT_DIR = Path("generated_angles")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Carregar o Pipeline

Usamos `QwenImageEditPlusPipeline` que suporta multiplas imagens de entrada nativamente.

In [ ]:
pipe = QwenImageEditPlusPipeline.from_pretrained(
    HF_BASE_MODEL,
    torch_dtype=DTYPE,
)
pipe = pipe.to(DEVICE)

# Otimizacoes de memoria
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

# Carregar LoRA de controle de angulos
pipe.load_lora_weights(LORA_ANGLES, adapter_name="angles")

print("Pipeline carregado com sucesso!")

## 4. Definir Angulos e Prompts

O LoRA de angulos entende comandos especificos para controle de camera.
Abaixo definimos presets uteis para fotografia de produtos em marketplaces.

In [ ]:
# Presets de angulos para fotografia de produtos
ANGLE_PRESETS = {
    "frontal":       "front view, eye-level shot, medium shot",
    "lateral_esq":   "left side view, eye-level shot, medium shot",
    "lateral_dir":   "right side view, eye-level shot, medium shot",
    "tres_quartos":  "three-quarter view, eye-level shot, medium shot",
    "top_down":      "top-down view, overhead shot",
    "angulo_baixo":  "bottom-up view, low-angle shot",
    "close_up":      "front view, eye-level shot, close-up shot",
    "wide_angle":    "front view, eye-level shot, wide shot",
    "costas":        "back view, eye-level shot, medium shot",
    "diagonal_45":   "rotate 45 degrees left, eye-level shot, medium shot",
}

# Backgrounds para e-commerce
BACKGROUND_PRESETS = {
    "estudio_branco":  "on a clean white studio background",
    "estudio_cinza":   "on a soft gray studio background",
    "marmore":         "on a marble surface with soft lighting",
    "lifestyle":       "in a lifestyle setting with natural lighting",
    "sem_fundo":       "",  # manter o fundo original
}


def compose_prompt(angle_key, background_key="estudio_branco", product_desc=""):
    """Monta o prompt final combinando angulo, fundo e descricao do produto."""
    angle = ANGLE_PRESETS[angle_key]
    bg = BACKGROUND_PRESETS.get(background_key, "")
    
    parts = [angle]
    if product_desc:
        parts.append(f"of the {product_desc}")
    if bg:
        parts.append(bg)
    
    return ", ".join(parts)

## 5. Funcao Principal de Geracao

Aceita uma lista de imagens (1 ou mais) e gera imagens em todos os angulos desejados.

In [ ]:
def generate_product_angles(
    images,
    angles=None,
    background="estudio_branco",
    product_desc="",
    num_inference_steps=40,
    true_cfg_scale=4.0,
    seed=42,
):
    """
    Gera imagens de um produto em multiplos angulos.
    
    Args:
        images: PIL.Image ou lista de PIL.Image do mesmo produto
        angles: lista de angulos desejados (chaves de ANGLE_PRESETS).
                Se None, usa um conjunto padrao para e-commerce.
        background: chave de BACKGROUND_PRESETS
        product_desc: descricao do produto (ex: "tenis branco Nike")
        num_inference_steps: passos de inferencia (mais = melhor qualidade, mais lento)
        true_cfg_scale: escala de guidance (4.0 recomendado)
        seed: seed para reproducibilidade
    
    Returns:
        dict: {angle_name: PIL.Image}
    """
    # Normalizar entrada para lista
    if isinstance(images, Image.Image):
        images = [images]
    
    # Angulos padrao para e-commerce
    if angles is None:
        angles = ["frontal", "lateral_esq", "lateral_dir", "tres_quartos", "top_down", "close_up"]
    
    results = {}
    
    for angle_key in angles:
        prompt = compose_prompt(angle_key, background, product_desc)
        print(f"  Gerando angulo '{angle_key}': {prompt}")
        
        output = pipe(
            image=images,
            prompt=prompt,
            negative_prompt=" ",
            true_cfg_scale=true_cfg_scale,
            num_inference_steps=num_inference_steps,
            guidance_scale=1.0,
            num_images_per_prompt=1,
            generator=torch.manual_seed(seed),
        )
        
        results[angle_key] = output.images[0]
    
    return results

## 6. Funcao para Visualizacao

In [ ]:
import matplotlib.pyplot as plt


def show_results(input_images, generated, product_name="Produto"):
    """Exibe as imagens de entrada e as geradas lado a lado."""
    n_inputs = len(input_images)
    n_generated = len(generated)
    total = n_inputs + n_generated
    cols = min(4, total)
    rows = (total + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]
    
    # Mostrar imagens de entrada
    for i, img in enumerate(input_images):
        axes[i].imshow(img)
        axes[i].set_title(f"Entrada {i+1}", fontsize=12, fontweight="bold")
        axes[i].axis("off")
    
    # Mostrar imagens geradas
    for i, (angle_name, img) in enumerate(generated.items()):
        idx = n_inputs + i
        axes[idx].imshow(img)
        axes[idx].set_title(f"{angle_name}", fontsize=12)
        axes[idx].axis("off")
    
    # Esconder eixos vazios
    for i in range(total, len(axes)):
        axes[i].axis("off")
    
    fig.suptitle(f"{product_name} - Angulos Gerados", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()


def save_results(generated, product_name="produto"):
    """Salva as imagens geradas no diretorio de saida."""
    saved = []
    for angle_name, img in generated.items():
        filename = f"{product_name}_{angle_name}.png"
        filepath = OUTPUT_DIR / filename
        img.save(filepath)
        saved.append(filepath)
        print(f"  Salvo: {filepath}")
    return saved

## 7. Exemplo: Uma Imagem de Entrada

Caso mais simples - uma unica foto do produto gera multiplos angulos.

In [ ]:
# Carregar uma imagem de produto (substitua pela URL ou caminho do seu produto)
produto_img = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png")

# Gerar angulos
print("Gerando angulos para produto (1 imagem de entrada)...")
results = generate_product_angles(
    images=produto_img,
    angles=["frontal", "lateral_esq", "lateral_dir", "top_down", "close_up"],
    background="estudio_branco",
    product_desc="product",
)

# Visualizar e salvar
show_results([produto_img], results, "Produto Exemplo")
save_results(results, "produto_exemplo")

## 8. Exemplo: Multiplas Imagens de Entrada

Quando voce tem 2+ fotos do mesmo produto (ex: frente e lateral),
o modelo usa todas como referencia para gerar angulos com mais consistencia.

In [ ]:
# Carregar multiplas imagens do mesmo produto
# Substitua pelos caminhos/URLs das suas imagens reais
img1 = load_image("produto_frente.png")   # foto frontal
img2 = load_image("produto_lateral.png")   # foto lateral
# img3 = load_image("produto_detalhe.png") # adicione quantas quiser

product_images = [img1, img2]

print(f"Gerando angulos com {len(product_images)} imagens de referencia...")
results = generate_product_angles(
    images=product_images,
    angles=["frontal", "tres_quartos", "costas", "top_down", "wide_angle"],
    background="estudio_branco",
    product_desc="product",
)

show_results(product_images, results, "Produto - Multi-Ref")
save_results(results, "produto_multi_ref")

## 9. Exemplo: Carregar Imagens de um Diretorio

Util para processar em batch - carrega todas as imagens de uma pasta como
referencias do mesmo produto.

In [ ]:
def load_product_images(directory):
    """Carrega todas as imagens de um diretorio como referencias do mesmo produto."""
    directory = Path(directory)
    extensions = {".png", ".jpg", ".jpeg", ".webp"}
    images = []
    for f in sorted(directory.iterdir()):
        if f.suffix.lower() in extensions:
            images.append(Image.open(f).convert("RGB"))
            print(f"  Carregada: {f.name}")
    print(f"Total: {len(images)} imagens carregadas")
    return images


# Exemplo de uso:
# product_images = load_product_images("./meu_produto/")
# results = generate_product_angles(product_images, product_desc="tenis branco")

## 10. Batch: Processar Multiplos Produtos

Para processar um catalogo inteiro, onde cada subpasta contem as fotos de um produto.

In [ ]:
def process_catalog(catalog_dir, angles=None, background="estudio_branco"):
    """
    Processa um catalogo de produtos.
    
    Estrutura esperada:
        catalog_dir/
            produto_1/
                foto1.jpg
                foto2.jpg
            produto_2/
                foto1.jpg
            ...
    """
    catalog_dir = Path(catalog_dir)
    all_results = {}
    
    for product_dir in sorted(catalog_dir.iterdir()):
        if not product_dir.is_dir():
            continue
        
        product_name = product_dir.name
        print(f"\nProcessando: {product_name}")
        
        images = load_product_images(product_dir)
        if not images:
            print(f"  Nenhuma imagem encontrada, pulando...")
            continue
        
        results = generate_product_angles(
            images=images,
            angles=angles,
            background=background,
            product_desc=product_name,
        )
        
        save_results(results, product_name)
        all_results[product_name] = results
    
    return all_results


# Exemplo de uso:
# all_results = process_catalog("./catalogo/")

## 11. Alternativa: Inferencia Remota (HF Inference Providers)

Se nao tiver GPU local, pode usar a API remota da Hugging Face.
Note que a API remota aceita apenas 1 imagem por chamada.

In [ ]:
# Token ja configurado na celula de Setup via Colab Secrets
# Caso nao tenha executado, descomente abaixo:
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import io
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)


def generate_angles_remote(image_path, angles=None, product_desc="product"):
    """
    Gera angulos usando a API remota (1 imagem de entrada).
    """
    if angles is None:
        angles = ["frontal", "lateral_esq", "lateral_dir", "top_down", "close_up"]
    
    with open(image_path, "rb") as f:
        input_bytes = f.read()
    
    results = {}
    for angle_key in angles:
        prompt = compose_prompt(angle_key, "estudio_branco", product_desc)
        print(f"  [{angle_key}] {prompt}")
        
        image = client.image_to_image(
            input_bytes,
            prompt=prompt,
            model="dx8152/Qwen-Edit-2509-Multiple-angles",
        )
        results[angle_key] = image
    
    return results


# Exemplo:
# results = generate_angles_remote("meu_produto.jpg", product_desc="tenis esportivo")
# save_results(results, "produto_remoto")